# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
%%writefile ../scripts/lane_features.py
"""Feature builder for my decline-risk lane (built in w03_feature_leakage_check, ML-05).
w04, w05, w06, w07 and the capstone import this so every notebook uses the exact same features.

Prediction moment = day -30 (start of the last 30 days). Only things known before that go in:
  days 61-90 back | days 31-60 back (prev_30d) || days 1-30 back = label window, never a feature
"""
import os
from pathlib import Path

import numpy as np
import pandas as pd

# joblib prints a long "could not find physical cores" warning on windows unless this is set
# below the logical core count (physical cores are usually half the logical ones)
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(max(1, (os.cpu_count() or 2) // 2)))

SEED = 42

RAW_CANDIDATES = [
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("content_refresh_anonymized.csv"),
]

NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "has_keyword_data",
    "word_count", "has_word_count",
    "age_at_prediction",
    "log_impr_prev30", "log_impr_d61_90",
    "log_clicks_prev30", "log_clicks_d61_90",
    "log_sessions_prev30", "log_sessions_d61_90",
    "impr_momentum_prior", "ctr_prev30", "prev30_empty",
]
CATEGORICAL_FEATURES = ["competition_level", "content_type", "main_intent"]


def load_raw() -> pd.DataFrame:
    path = next(p for p in RAW_CANDIDATES if p.exists())
    return pd.read_csv(path)


def build_features(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.copy()
    df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

    # back out days 61-90 (the windows nest exactly, checked in the w03 data contract)
    df["impr_d61_90"] = df["impressions_90d"] - df["impressions_last_30d"] - df["impressions_prev_30d"]
    df["clicks_d61_90"] = df["clicks_90d"] - df["clicks_last_30d"] - df["clicks_prev_30d"]
    df["sessions_d61_90"] = df["sessions_90d"] - df["sessions_last_30d"] - df["sessions_prev_30d"]

    # traffic is heavy tailed, so log1p it
    df["log_impr_prev30"] = np.log1p(df["impressions_prev_30d"])
    df["log_impr_d61_90"] = np.log1p(df["impr_d61_90"])
    df["log_clicks_prev30"] = np.log1p(df["clicks_prev_30d"])
    df["log_clicks_d61_90"] = np.log1p(df["clicks_d61_90"])
    df["log_sessions_prev30"] = np.log1p(df["sessions_prev_30d"])
    df["log_sessions_d61_90"] = np.log1p(df["sessions_d61_90"])

    # momentum going INTO day -30: prev_30d vs the 30 days before it (log ratio, +1 so zeros dont blow up)
    df["impr_momentum_prior"] = np.log((df["impressions_prev_30d"] + 1) / (df["impr_d61_90"] + 1))
    df["ctr_prev30"] = np.where(
        df["impressions_prev_30d"] > 0, df["clicks_prev_30d"] / df["impressions_prev_30d"] * 100, 0.0
    )
    # nothing to drop from -> can never be labelled down (structural, known at day -30)
    df["prev30_empty"] = (df["impressions_prev_30d"] == 0).astype(int)

    # missing keyword / length data follows content_type, so flag it first, then fill
    df["has_keyword_data"] = df["search_volume"].notna().astype(int)
    df["has_word_count"] = df["word_count"].notna().astype(int)
    for col in ["search_volume", "competition", "cpc", "word_count"]:
        df[col] = df[col].fillna(0)

    df["age_at_prediction"] = df["content_age_days"] - 30
    for col in CATEGORICAL_FEATURES:
        df[col] = df[col].fillna("unknown").astype(str)
    return df


def feature_matrix(df: pd.DataFrame) -> pd.DataFrame:
    numeric = df[NUMERIC_FEATURES].astype(float)
    dummies = pd.get_dummies(df[CATEGORICAL_FEATURES], dtype=float)
    return pd.concat([numeric, dummies], axis=1)


def client_holdout(df: pd.DataFrame, test_share: float = 0.2, seed: int = SEED):
    """Same idea as scripts/03_train_model.py: shuffle the clients with seed 42, hold out 20% of them.
    Returns boolean masks (train, test) so no client is on both sides."""
    clients = df["client_id"].drop_duplicates().to_numpy()
    shuffled = np.random.default_rng(seed).permutation(clients)
    n_test = max(1, int(round(len(shuffled) * test_share)))
    test_mask = df["client_id"].isin(set(shuffled[:n_test])).to_numpy()
    return ~test_mask, test_mask


def precision_at_k(y, scores, k: int) -> float:
    order = np.argsort(-np.asarray(scores, dtype=float), kind="stable")[:k]
    return float(np.asarray(y)[order].mean())

Overwriting ../scripts/lane_features.py


In [2]:
"""Using the file I just wrote. Build the vector from the raw starter CSV and sanity check it: shape, no NaNs left, and the backed-out days 61-90 windows never go negative."""

import sys
sys.path.insert(0, "../scripts")
import pandas as pd
import lane_features as lf

raw = lf.load_raw()
df = lf.build_features(raw)
X = lf.feature_matrix(df)
y = df["is_declining_label"].to_numpy()

print("rows:", len(df), "| numeric:", len(lf.NUMERIC_FEATURES), "| categorical:", len(lf.CATEGORICAL_FEATURES), "| columns after one-hot:", X.shape[1])
print("NaNs left in X:", int(X.isna().sum().sum()))
print("negative days 61-90 values:", int((df[["impr_d61_90", "clicks_d61_90", "sessions_d61_90"]] < 0).sum().sum()))
print("base rate (is_declining_label):", round(y.mean(), 3))
X.head()

rows: 30000 | numeric: 16 | categorical: 3 | columns after one-hot: 28
NaNs left in X: 0
negative days 61-90 values: 0
base rate (is_declining_label): 0.542


,search_volume,competition,cpc,has_keyword_data,word_count,has_word_count,age_at_prediction,log_impr_prev30,log_impr_d61_90,log_clicks_prev30,...,competition_level_MEDIUM,competition_level_unknown,content_type_comparison article,content_type_feedly article,content_type_keyword article,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional,main_intent_unknown
0,10.0,0.67,2.05,1.0,3221.0,1.0,157.0,6.895683,7.713785,2.639057,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,90.0,0.01,0.05,1.0,2481.0,1.0,415.0,8.685416,8.840001,0.693147,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
2,0.0,0.00,0.00,1.0,3515.0,1.0,111.0,8.714403,8.321422,1.386294,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
3,10.0,0.00,0.00,1.0,0.0,0.0,433.0,8.344505,8.273847,2.890372,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
4,0.0,0.00,0.00,1.0,2803.0,1.0,233.0,8.772300,9.045230,1.098612,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [3]:
"""One line per feature: what it means, how blanks are handled, and when it exists. The test for "available when" is simple, it has to be known before day -30. word_count is the only soft one (measured at export, so an edit inside the label window could change it), I keep it but note it."""

notes = pd.DataFrame([
    ("search_volume", "keyword demand estimate", "blank -> 0 + has_keyword_data", "content metadata, before day -30"),
    ("competition", "keyword competition 0-1", "blank -> 0 + has_keyword_data", "content metadata, before day -30"),
    ("cpc", "keyword cost per click", "blank -> 0 + has_keyword_data", "content metadata, before day -30"),
    ("has_keyword_data", "1 = page has keyword data", "never blank", "content metadata, before day -30"),
    ("word_count", "article length", "blank -> 0 + has_word_count", "measured at export (soft, see above)"),
    ("has_word_count", "1 = word count was measured", "never blank", "measured at export"),
    ("age_at_prediction", "content_age_days - 30", "never blank", "before day -30 by construction"),
    ("log_impr_prev30", "log1p impressions, days 31-60", "never blank", "before day -30"),
    ("log_impr_d61_90", "log1p impressions, days 61-90 (90d - last - prev)", "never blank, never < 0", "before day -30"),
    ("log_clicks_prev30", "log1p clicks, days 31-60", "never blank", "before day -30"),
    ("log_clicks_d61_90", "log1p clicks, days 61-90", "never blank, never < 0", "before day -30"),
    ("log_sessions_prev30", "log1p GA4 sessions, days 31-60", "never blank", "before day -30"),
    ("log_sessions_d61_90", "log1p GA4 sessions, days 61-90", "never blank, never < 0", "before day -30"),
    ("impr_momentum_prior", "log((prev30+1)/(d61_90+1)), rising or falling into day -30", "never blank", "before day -30"),
    ("ctr_prev30", "clicks/impressions days 31-60, x100 scale like ctr", "0 when prev impressions = 0", "before day -30"),
    ("prev30_empty", "1 = zero impressions days 31-60 (cant be labelled down)", "never blank", "before day -30"),
    ("competition_level", "LOW / MEDIUM / HIGH", "blank -> 'unknown' category", "content metadata, before day -30"),
    ("content_type", "keyword / feedly / comparison article", "never blank", "content metadata, before day -30"),
    ("main_intent", "search intent of the target keyword", "blank -> 'unknown' category", "content metadata, before day -30"),
], columns=["feature", "meaning", "missing handling", "available when"])

raw_source = {"search_volume": "search_volume", "competition": "competition", "cpc": "cpc", "word_count": "word_count",
              "competition_level": "competition_level", "main_intent": "main_intent", "content_type": "content_type"}
notes["raw_blank_%"] = notes["feature"].map(lambda f: round(raw[raw_source[f]].isna().mean() * 100, 1) if f in raw_source else 0.0)
display(notes)

# why has_ flags and not a blind fillna(0): blanks follow content_type, and the decline rate differs with them
check = raw.assign(keyword_blank=raw["search_volume"].isna(), word_count_blank=raw["word_count"].isna(),
                   declining=raw["trend_direction"].eq("down"))
display(check.groupby("content_type")[["keyword_blank", "word_count_blank"]].mean().round(3))
display(check.groupby("word_count_blank")["declining"].agg(["mean", "size"]).round(3))

,feature,meaning,missing handling,available when,raw_blank_%
0,search_volume,keyword demand estimate,blank -> 0 + has_keyword_data,"content metadata, before day -30",8.2
1,competition,keyword competition 0-1,blank -> 0 + has_keyword_data,"content metadata, before day -30",8.2
2,cpc,keyword cost per click,blank -> 0 + has_keyword_data,"content metadata, before day -30",8.2
3,has_keyword_data,1 = page has keyword data,never blank,"content metadata, before day -30",0.0
4,word_count,article length,blank -> 0 + has_word_count,"measured at export (soft, see above)",25.7
5,has_word_count,1 = word count was measured,never blank,measured at export,0.0
6,age_at_prediction,content_age_days - 30,never blank,before day -30 by construction,0.0
7,log_impr_prev30,"log1p impressions, days 31-60",never blank,before day -30,0.0
8,log_impr_d61_90,"log1p impressions, days 61-90 (90d - last - prev)","never blank, never < 0",before day -30,0.0
9,log_clicks_prev30,"log1p clicks, days 31-60",never blank,before day -30,0.0


,keyword_blank,word_count_blank
content_type,,
comparison article,0.000,0.000
feedly article,1.000,0.000
keyword article,0.014,0.283


,mean,size
word_count_blank,,
False,0.569,22301
True,0.465,7699


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [4]:
"""Attacking my own features. Same quick harness every time (gradient boosting, client holdout seed 42 = 20% of clients never seen in training) so only the feature set changes between rows. Its a test bench, not my final model.
1. harness check: add trend_pct on purpose. If the score doesnt jump toward 1.0 my harness is broken.
2. window arithmetic: add the raw impressions_90d next to my windows. last_30d = 90d - prev - d61_90 so the model can rebuild the label by subtraction.
3. overlap: add the 90 day rates the reference pipeline uses (ctr, avg_position, days_with_impressions).
4. timing: add days_since_last_update (68% of updates happen inside the label window).
5. control: shuffle the training labels, should fall to ~0.5 AUC. One shuffle is noisy on 6 test clients so I average 5.
Base rate of the held-out clients is printed next to every number."""

import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

train, test = lf.client_holdout(df)
print(f"train rows {train.sum():,} | test rows {test.sum():,} from {df.loc[test, 'client_id'].nunique()} unseen clients | test base rate {y[test].mean():.3f}")

def harness(extra=(), shuffle_seed=None):
    Xh = X.copy()
    for col in extra:
        Xh[col] = pd.to_numeric(raw[col], errors="coerce").fillna(0)
    y_train = y[train].copy()
    if shuffle_seed is not None:
        y_train = np.random.default_rng(shuffle_seed).permutation(y_train)
    model = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=200, random_state=lf.SEED)
    model.fit(Xh[train], y_train)
    s = model.predict_proba(Xh[test])[:, 1]
    return {"AUC": roc_auc_score(y[test], s), "P@50": lf.precision_at_k(y[test], s, 50), "P@200": lf.precision_at_k(y[test], s, 200)}

runs = {
    "my as-of features": harness(),
    "+ trend_pct (harness check)": harness(["trend_pct"]),
    "+ impressions_90d (window arithmetic)": harness(["impressions_90d"]),
    "+ ctr, avg_position, days_with_impressions (90d overlap)": harness(["ctr", "avg_position", "days_with_impressions"]),
    "+ days_since_last_update (timing)": harness(["days_since_last_update"]),
    "shuffled training labels (control, mean of 5)": pd.DataFrame([harness(shuffle_seed=s) for s in range(5)]).mean().to_dict(),
}
hunt = pd.DataFrame(runs).T.round(3)
hunt["test base rate"] = round(y[test].mean(), 3)
display(hunt)

# no single feature of mine should tower on its own (univariate AUC, train rows only, direction-free)
uni = {c: roc_auc_score(y[train], X.loc[train, c]) for c in lf.NUMERIC_FEATURES}
uni = pd.Series({c: max(a, 1 - a) for c, a in uni.items()}).sort_values(ascending=False).round(3)
print("top univariate AUCs of my features:")
print(uni.head(5).to_string())

# ids and product flags
print("id columns in X:", [c for c in X.columns if "id" in c.lower().split("_")])
print("product-flag style columns in the raw file:", [c for c in raw.columns if any(k in c for k in ["health", "needs_", "quick_win", "score", "flag"])])

mine = hunt["AUC"].iloc[0]
print(f"""
verdict:
- harness works, trend_pct takes AUC to {hunt['AUC'].iloc[1]:.3f}
- the 90d leak is real, not a theory: raw impressions_90d alone takes AUC from {mine:.3f} to {hunt['AUC'].iloc[2]:.3f}, the 90d rates to {hunt['AUC'].iloc[3]:.3f}. thats the label window talking, so they stay out
- days_since_last_update only moves it a little ({hunt['AUC'].iloc[4]:.3f}) but its still measured after day -30, out on principle
- shuffled labels land at {hunt['AUC'].iloc[5]:.3f}, so my as-of score is not a harness artifact
- nothing in my own set towers (best single feature AUC {uni.iloc[0]:.3f}), no id columns, no product flags in the file""")

train rows 27,675 | test rows 2,325 from 6 unseen clients | test base rate 0.391


,AUC,P@50,P@200,test base rate
my as-of features,0.854,0.740,0.785,0.391
+ trend_pct (harness check),1.000,1.000,1.000,0.391
+ impressions_90d (window arithmetic),0.967,1.000,1.000,0.391
"+ ctr, avg_position, days_with_impressions (90d overlap)",0.936,1.000,0.955,0.391
+ days_since_last_update (timing),0.870,0.780,0.815,0.391
"shuffled training labels (control, mean of 5)",0.499,0.524,0.570,0.391


top univariate AUCs of my features:
prev30_empty         0.604
log_impr_prev30      0.599
log_impr_d61_90      0.593
age_at_prediction    0.589
word_count           0.560
id columns in X: []
product-flag style columns in the raw file: []

verdict:
- harness works, trend_pct takes AUC to 1.000
- the 90d leak is real, not a theory: raw impressions_90d alone takes AUC from 0.854 to 0.967, the 90d rates to 0.936. thats the label window talking, so they stay out
- days_since_last_update only moves it a little (0.870) but its still measured after day -30, out on principle
- shuffled labels land at 0.499, so my as-of score is not a harness artifact
- nothing in my own set towers (best single feature AUC 0.604), no id columns, no product flags in the file


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [5]:
"""What I refused to use, one line of why each. Same list as the w03 data contract, here its the final word for the feature vector. The theme is the same for almost all of them: they were measured over a window that includes the label window, so at day -30 they didnt exist yet."""

refused = {
    "trend_direction / trend_pct": "the label and its source, test 1 above shows what they do",
    "impressions_last_30d / clicks_last_30d / sessions_last_30d": "measured inside the label window",
    "impressions_90d / clicks_90d / sessions_90d (raw)": "contain the label window, test 2 shows the subtraction leak; only used to back out days 61-90",
    "pageviews_90d, users_90d, engaged_sessions_90d, ai_sessions_90d, scroll_events_90d": "90d totals incl. label window, no 30d split to back them out",
    "days_with_impressions, days_with_sessions": "counted over 90d incl. label window",
    "ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct": "90d rates incl. label window (ctr_prev30 replaces ctr)",
    "impression_tier, position_tier": "buckets of 90d columns, same overlap",
    "days_since_last_update, freshness_tier": "68% updated inside the label window, and a refresh can be a reaction to the drop",
    "provider_used, model_used": "generation metadata, the dictionary says not a feature, ~71% blank",
    "char_count, char_count_tier, word_count_tier": "near duplicates of word_count",
    "age_tier, age_tier_order": "buckets of content age, redundant",
    "content_id, client_id": "pseudonyms, used only to split by client and join",
}
for cols, why in refused.items():
    print(f"- {cols}: {why}")
print()
print(f"final feature vector: {len(lf.NUMERIC_FEATURES)} numeric + {len(lf.CATEGORICAL_FEATURES)} categorical = {X.shape[1]} columns after one-hot")
print("numeric:", ", ".join(lf.NUMERIC_FEATURES))

- trend_direction / trend_pct: the label and its source, test 1 above shows what they do
- impressions_last_30d / clicks_last_30d / sessions_last_30d: measured inside the label window
- impressions_90d / clicks_90d / sessions_90d (raw): contain the label window, test 2 shows the subtraction leak; only used to back out days 61-90
- pageviews_90d, users_90d, engaged_sessions_90d, ai_sessions_90d, scroll_events_90d: 90d totals incl. label window, no 30d split to back them out
- days_with_impressions, days_with_sessions: counted over 90d incl. label window
- ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct: 90d rates incl. label window (ctr_prev30 replaces ctr)
- impression_tier, position_tier: buckets of 90d columns, same overlap
- days_since_last_update, freshness_tier: 68% updated inside the label window, and a refresh can be a reaction to the drop
- provider_used, model_used: generation metadata, the dictionary says not a feature, ~71% blank
- char_count, char_count_tier

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.